# TTPTrace — İlk Oturum Analizi

Bu notebook, ilk kontrollü benign oturumun JSONL verilerini doğrular, ağ olaylarını süreç oluşturma kayıtlarına bağlar ve süreç bazında ağ özellikleri üretir.

## Çalıştırma

Colab'da CPU çalışma ortamı yeterlidir. Hücreleri yukarıdan aşağıya çalıştırın. Dosyalar panelinden `events.jsonl`, `analysis-summary.json`, `manifest.json` ve `process-context.jsonl` dosyalarını `/content` dizinine yükleyin. Hücrelerdeki dosya seçici çalışmazsa sol Dosyalar panelini kullanın.

Bu sürüm `TTPT-20260929-B001` oturumuna özgüdür: SHA-256 değerleri, beklenen sayılar ve örnek süreç GUID'i bu oturuma aittir. Farklı veriler için bu kontrollerin uyarlanması gerekir. Ham veriler kamuya açık depoda bulunmaz; notebook tek başına örnek veri sağlamaz.

## Analiz kapsamı

- 269 oturum olayı: 23 süreç oluşturma, 194 ağ ve 15 DNS olayı dahil.
- Oturum içindeki süreç kayıtları 53 ağ olayını açıklar. Dört oturum öncesi süreç kaydı 139 ek eşleşme sağlar; iki olay çözümlenmemiş olarak korunur.
- Ağ olayı bulunan altı süreç için olay sayısı, farklı hedef IP/port sayıları, TCP/UDP sayıları ve oranları hesaplanır.
- Saniye başına değer, tüm oturum penceresindeki ortalama kayıt sıklığıdır; paket veya bayt ölçümü değildir.
- `network-process-links.jsonl` ve `network-process-features.jsonl` oluşturulur ve tekrar okunarak doğrulanır.

Süreç kimlikleri ve yollar izlenebilirlik içindir; doğrudan model girdisi olarak seçilmemiştir. Bu tek oturum, model eğitimi veya başarı değerlendirmesi için yeterli değildir. DNS özellikleri ve model eğitimi sonraki aşamalardır.

## Saklama

Notebook'un GitHub kopyasında çıktılar ve çalışma ortamı metaverileri temizlenmiştir. Kendi verilerinizle çalıştırınca çıktılarda ortam bilgileri görünebilir. Colab `/content` dosyaları geçicidir; üretilen dosyaları indirip yerel `data/processed/` altında saklayın. Bu veriler sürüm kontrolünün dışındadır.


In [ ]:
# 1. Dosya konumları ve yeniden yükleme
import json
import hashlib
import sys
from pathlib import Path
from collections import Counter

DATA_DIR = Path("/content")
REQUIRED_FILES = ["events.jsonl", "analysis-summary.json", "manifest.json"]

missing = [name for name in REQUIRED_FILES if not (DATA_DIR / name).is_file()]
if missing:
    from google.colab import files
    print("Eksik dosyaları seç:", ", ".join(missing))
    files.upload()

missing = [name for name in REQUIRED_FILES if not (DATA_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f"Eksik dosyalar: {missing}")

print("Python:", sys.version.split()[0])
for name in REQUIRED_FILES:
    print(f"Hazır: {name} ({(DATA_DIR / name).stat().st_size} bayt)")

In [ ]:
# 2. Aktarım bütünlüğü ve oturum bilgisi
# Bunlar bu oturumun yerel dosyalarından hesaplanan SHA-256 değerleridir.
EXPECTED_SHA256 = {
    "events.jsonl": "040a2a1174aa2c169ace386d253000ab8e73fb14cb135fd7839153abcadaf8e4",
    "analysis-summary.json": "62a42d1b8e476ce6424f33d6f0425b646f45d229b35dbd33a8d519d33200efad",
    "manifest.json": "b2b256cfc37a0a8f44afdf250b3cc4a6f62b7fbafc5b81e526e6fb25cf7edd91",
}

for name, expected in EXPECTED_SHA256.items():
    actual = hashlib.sha256((DATA_DIR / name).read_bytes()).hexdigest()
    if actual != expected:
        raise ValueError(f"Dosya bütünlüğü uyuşmuyor: {name}")
    print(f"SHA-256 doğrulandı: {name}")

manifest = json.loads((DATA_DIR / "manifest.json").read_text(encoding="utf-8"))
summary = json.loads((DATA_DIR / "analysis-summary.json").read_text(encoding="utf-8"))
if manifest["experiment_id"] != summary["experiment_id"]:
    raise ValueError("Manifest ile analiz özeti farklı oturumlara ait.")

print("Deney:", manifest["experiment_id"])
print("Oturum türü:", manifest["session_type"])

In [ ]:
# 3. JSONL dosyasını olay listesine dönüştürme
def load_events(path: Path) -> list[dict]:
    events = []
    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            try:
                event = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"Satır {line_number}: geçersiz JSON") from exc
            if not isinstance(event, dict):
                raise ValueError(f"Satır {line_number}: JSON nesnesi bekleniyordu")
            events.append(event)
    return events

events = load_events(DATA_DIR / "events.jsonl")
print("Olay sayısı:", len(events))
print("İlk olayın alanları:", ", ".join(events[0].keys()) if events else "Dosya boş")

In [ ]:
# 4. Önceki analizle sayısal karşılaştırma
counts = Counter(event["event_id"] for event in events)
expected_counts = {1: 23, 3: 194, 5: 28, 11: 3, 12: 3, 13: 3, 22: 15}

checks = {
    "Toplam olay": len(events) == summary["analyzed_event_count"] == 269,
    "Ağ olayları": counts[3] == summary["network_event_count"] == 194,
    "DNS olayları": counts[22] == summary["dns_event_count"] == 15,
    "Olay türü dağılımı": dict(counts) == expected_counts,
    "Oturum kimlikleri": all(
        event["experiment_id"] == manifest["experiment_id"]
        and event["host_id"] == manifest["host"]["host_id"]
        for event in events
    ),
}

for event_id, count in sorted(counts.items()):
    print(f"Event ID {event_id:>2}: {count}")
print()
for name, passed in checks.items():
    print(f"{name}: {'OK' if passed else 'UYUŞMUYOR'}")

if not all(checks.values()):
    raise ValueError("Kontrollerden biri uyuşmuyor; çıktıyı incele.")

print("\nColab aktarımı ve başlangıç kontrolleri tamamlandı.")

In [ ]:
def filter_events(events: list[dict], event_id: int) -> list[dict]:
    selected = []

    for event in events:
        if event["event_id"] == event_id:
            selected.append(event)

    return selected


process_events = filter_events(events, 1)
network_events = filter_events(events, 3)
dns_events = filter_events(events, 22)

print("Süreç oluşturma:", len(process_events))
print("Ağ olayları:", len(network_events))
print("DNS olayları:", len(dns_events))

In [ ]:
def build_process_index(process_events: list[dict]) -> dict[str, dict]:
    process_index = {}

    for event in process_events:
        process_guid = event["data"]["ProcessGuid"]

        if process_guid in process_index:
            raise ValueError(f"Tekrarlanan ProcessGuid: {process_guid}")

        process_index[process_guid] = event

    return process_index


process_index = build_process_index(process_events)

whoami_guid = "{14bdcb74-dd88-6aba-1c01-000000000600}"
whoami_event = process_index[whoami_guid]

parent_guid = whoami_event["data"]["ParentProcessGuid"]
parent_event = process_index.get(parent_guid)

print("İndeksteki süreç sayısı:", len(process_index))
print("Çocuk süreç:", whoami_event["data"]["Image"])
print("Ebeveyn GUID:", parent_guid)

if parent_event is None:
    print("Ebeveynin oluşturma kaydı bu oturumun içinde bulunamadı.")
else:
    print("Ebeveyn süreç:", parent_event["data"]["Image"])
    print("Ebeveyn komutu:", parent_event["data"]["CommandLine"])

In [ ]:
def link_events_to_processes(
    events: list[dict],
    process_index: dict[str, dict],
) -> list[dict]:
    linked_events = []

    for event in events:
        process_guid = event["data"]["ProcessGuid"]
        process_event = process_index.get(process_guid)

        linked_events.append({
            "record_id": event["record_id"],
            "utc_time": event["utc_time"],
            "process_guid": process_guid,
            "image": event["data"]["Image"],
            "process_creation_found": process_event is not None,
            "command_line": (
                process_event["data"]["CommandLine"]
                if process_event is not None
                else None
            ),
        })

    return linked_events


network_links = link_events_to_processes(network_events, process_index)

matched_count = sum(
    link["process_creation_found"] for link in network_links
)
unmatched_count = len(network_links) - matched_count

print("Toplam ağ olayı:", len(network_links))
print("Süreç oluşturma kaydı bulunan:", matched_count)
print("Süreç oluşturma kaydı bulunamayan:", unmatched_count)
print("Sayı kontrolü:", matched_count + unmatched_count == len(network_events))

matched_example = next(
    (link for link in network_links if link["process_creation_found"]),
    None,
)

if matched_example is not None:
    print("\nEşleşen bir örnek:")
    print(json.dumps(matched_example, indent=2, ensure_ascii=False))

In [ ]:
unmatched_links = [
    link for link in network_links
    if not link["process_creation_found"]
]

unmatched_process_counts = Counter(
    link["process_guid"] for link in unmatched_links
)

unmatched_images = {
    link["process_guid"]: link["image"]
    for link in unmatched_links
}

print("Eşleşmeyen ağ olayı:", len(unmatched_links))
print("Eşleşmeyen farklı süreç:", len(unmatched_process_counts))

for process_guid, count in unmatched_process_counts.most_common():
    print(f"\nOlay sayısı: {count}")
    print("ProcessGuid:", process_guid)
    print("Program:", unmatched_images[process_guid])

In [ ]:
from datetime import datetime
from google.colab import files

context_path = DATA_DIR / "process-context.jsonl"

if not context_path.is_file():
    print("process-context.jsonl dosyasını seç.")
    files.upload()

context_events = load_events(context_path)
session_start = datetime.fromisoformat(
    manifest["window"]["started_at"].replace("Z", "+00:00")
)

for event in context_events:
    if event["event_id"] != 1:
        raise ValueError("Bağlam dosyasında süreç oluşturma olayı bekleniyordu.")

    if (
        event["experiment_id"] != manifest["experiment_id"]
        or event["host_id"] != manifest["host"]["host_id"]
    ):
        raise ValueError("Bağlam kaydı farklı bir oturuma veya bilgisayara ait.")

    context_time = datetime.fromisoformat(event["utc_time"])

    if context_time >= session_start:
        raise ValueError("Bağlam kaydı oturum başlangıcından önce değil.")

full_process_index = build_process_index(
    process_events + context_events
)

extended_network_links = link_events_to_processes(
    network_events,
    full_process_index,
)

extended_matched_count = sum(
    link["process_creation_found"]
    for link in extended_network_links
)

remaining_links = [
    link for link in extended_network_links
    if not link["process_creation_found"]
]

print("Eklenen bağlam kaydı:", len(context_events))
print("Genişletilmiş süreç indeksi:", len(full_process_index))
print("Toplam ağ olayı:", len(extended_network_links))
print("Eşleşen ağ olayı:", extended_matched_count)
print("Eşleşmeyen ağ olayı:", len(remaining_links))
print(
    "Sayı kontrolü:",
    extended_matched_count + len(remaining_links) == len(network_events),
)

for process_guid, count in Counter(
    link["process_guid"] for link in remaining_links
).items():
    print(f"Kalan süreç: {process_guid} — {count} olay")

In [ ]:
network_process_links = []

for link in extended_network_links:
    process_guid = link["process_guid"]
    process_event = full_process_index.get(process_guid)

    if process_guid in process_index:
        source = "session"
    elif process_event is not None:
        source = "pre_session"
    else:
        source = "unresolved"

    enriched_link = link.copy()
    enriched_link["process_creation_source"] = source
    enriched_link["process_creation_record_id"] = (
        process_event["record_id"]
        if process_event is not None
        else None
    )
    enriched_link["process_creation_utc_time"] = (
        process_event["utc_time"]
        if process_event is not None
        else None
    )

    network_process_links.append(enriched_link)

output_path = DATA_DIR / "network-process-links.jsonl"

with output_path.open("w", encoding="utf-8", newline="\n") as file:
    for link in network_process_links:
        file.write(json.dumps(link, ensure_ascii=False) + "\n")

source_counts = Counter(
    link["process_creation_source"]
    for link in network_process_links
)

print("Kaydedilen bağlantı kaydı:", len(network_process_links))

for source, count in sorted(source_counts.items()):
    print(f"{source}: {count}")

print("Dosya:", output_path)

In [ ]:
from google.colab import files

saved_links = load_events(output_path)

original_record_counts = Counter(
    event["record_id"] for event in network_events
)

saved_record_counts = Counter(
    link["record_id"] for link in saved_links
)

if original_record_counts != saved_record_counts:
    raise ValueError(
        "Kaydedilen bağlantılarda eksik, fazla veya tekrarlanan kayıt var."
    )

saved_source_counts = Counter(
    link["process_creation_source"] for link in saved_links
)

if saved_source_counts != {
    "session": 53,
    "pre_session": 139,
    "unresolved": 2,
}:
    raise ValueError("Kaydedilen süreç bağlamı dağılımı uyuşmuyor.")

print("Dosya doğrulandı:", len(saved_links), "kayıt")
files.download(str(output_path))

In [ ]:
def summarize_network_by_process(network_events: list[dict]) -> list[dict]:
    groups = {}

    for event in network_events:
        process_guid = event["data"]["ProcessGuid"]
        groups.setdefault(process_guid, []).append(event)

    features = []

    for process_guid, process_events in groups.items():
        destination_ips = {
            event["data"]["DestinationIp"]
            for event in process_events
        }

        destination_ports = {
            int(event["data"]["DestinationPort"])
            for event in process_events
        }

        protocols = Counter(
            event["data"]["Protocol"].lower()
            for event in process_events
        )

        features.append({
            "process_guid": process_guid,
            "image": process_events[0]["data"]["Image"],
            "network_event_count": len(process_events),
            "unique_destination_ip_count": len(destination_ips),
            "unique_destination_port_count": len(destination_ports),
            "tcp_event_count": protocols["tcp"],
            "udp_event_count": protocols["udp"],
        })

    return sorted(
        features,
        key=lambda feature: feature["network_event_count"],
        reverse=True,
    )


network_features = summarize_network_by_process(network_events)

print("Özetlenen süreç:", len(network_features))
print(
    "Toplam ağ olayı:",
    sum(feature["network_event_count"] for feature in network_features),
)

print("\nEn fazla ağ olayı üreten sürecin özellikleri:")
print(json.dumps(network_features[0], indent=2, ensure_ascii=False))

In [ ]:
from datetime import datetime

window_start = datetime.fromisoformat(
    manifest["window"]["started_at"].replace("Z", "+00:00")
)
window_end = datetime.fromisoformat(
    manifest["window"]["ended_at"].replace("Z", "+00:00")
)

window_duration_seconds = (window_end - window_start).total_seconds()

if window_duration_seconds <= 0:
    raise ValueError("Oturum süresi sıfırdan büyük olmalı.")

enriched_network_features = []

for feature in network_features:
    event_count = feature["network_event_count"]

    if event_count <= 0:
        raise ValueError("Ağ özeti en az bir olay içermeli.")

    enriched_feature = feature.copy()
    enriched_feature["tcp_event_ratio"] = (
        feature["tcp_event_count"] / event_count
    )
    enriched_feature["udp_event_ratio"] = (
        feature["udp_event_count"] / event_count
    )
    enriched_feature["network_events_per_window_second"] = (
        event_count / window_duration_seconds
    )

    enriched_network_features.append(enriched_feature)

print("Oturum süresi:", round(window_duration_seconds, 3), "saniye")
print("Özetlenen süreç:", len(enriched_network_features))
print("\nİlk sürecin genişletilmiş özellikleri:")
print(json.dumps(enriched_network_features[0], indent=2, ensure_ascii=False))

In [ ]:
feature_records = []

for feature in enriched_network_features:
    record = {
        "schema_version": "0.1.0",
        "experiment_id": manifest["experiment_id"],
        "host_id": manifest["host"]["host_id"],
        "window_started_at": window_start.isoformat(),
        "window_ended_at": window_end.isoformat(),
        "window_duration_seconds": window_duration_seconds,
        **feature,
    }
    feature_records.append(record)

total_network_events = sum(
    record["network_event_count"]
    for record in feature_records
)

if total_network_events != len(network_events):
    raise ValueError("Özetlerdeki olay toplamı kaynak kayıtlarla eşleşmiyor.")

feature_path = DATA_DIR / "network-process-features.jsonl"

with feature_path.open("w", encoding="utf-8", newline="\n") as file:
    for record in feature_records:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")

saved_features = load_events(feature_path)

if saved_features != feature_records:
    raise ValueError("Kaydedilen özellikler bellekteki kayıtlarla eşleşmiyor.")

print("Kaydedilen süreç özeti:", len(saved_features))
print("Temsil edilen ağ olayı:", total_network_events)
print("Dosya doğrulandı:", saved_features == feature_records)
print("Dosya:", feature_path)

In [ ]:
from google.colab import files

files.download(str(feature_path))